# Cross-dataset raw persona audit

## Excluded persona attributes

Attributes intentionally omitted from `persona_spec.yaml`:

| Attribute | Reason for exclusion |
| --- | --- |
| `census_division` | Excluded as a separate attribute (only natively present in `issp`). ISSP's 9 US Census divisions are kept directly as union levels inside `region` without collapsing. |
| `interview_language` | Excluded (`issp` `INTLANG`, `wvs7` `S_INTLANGUAGE`): survey-administration technical variable (~98% English), not a respondent persona trait. |
| `birth_year` | Excluded (`issp` `BIRTH`, `wvs7` `Q261`): 100% redundant with locked `age`, and pairing `age` with exact `birth_year` leaks survey wave year. |
| `US_DEGR` | Excluded (`issp` 2015-2019): 100% redundant with locked `education` (`EDULEVEL`/`DEGREE`). |
| `X003R` | Excluded (`wvs7`): 6-bin age recode, 100% redundant with locked exact `age` (`Q262`). |
| `EDUCYRS` | Excluded (`issp` years of full-time schooling): covered by locked `education`. |
| `TOPBOT` | Excluded (`issp` 1-10 subjective social standing scale): omitted per user review. |
| `WORK` | Excluded (`issp` paid work history): largely redundant with `employment_status`; omitted per user review. |
| `WRKHRS` | Excluded (`issp` weekly hours worked): omitted per user review. |
| `EMPREL` | Excluded (`issp` employment relationship): omitted per user review. |
| `WRKSUP` | Excluded (`issp` supervises other employees): omitted per user review. |
| `NSUP` | Excluded (`issp` number of employees supervised): omitted per user review. |
| `TYPORG1` | Excluded (`issp` for-profit vs non-profit organization): omitted per user review. |
| `PARTLIV` | Excluded (`issp` living in steady partnership): omitted per user review. |
| `SPWORK` | Excluded (`issp` spouse/partner paid work history): omitted per user review. |
| `SPWRKHRS` | Excluded (`issp` spouse/partner weekly hours worked): omitted per user review. |
| `SPEMPREL` | Excluded (`issp` spouse/partner employment relationship): omitted per user review. |
| `SPWRKSUP` | Excluded (`issp` spouse/partner supervises other employees): omitted per user review. |
| `SPISCO08` | Excluded (`issp` spouse/partner ISCO-08 occupation): omitted per user review. |
| `HHCHILDR` | Excluded (`issp` school-age children in household): omitted per user review. |
| `HHTODD` | Excluded (`issp` toddlers below school entry age in household): omitted per user review. |
| `HHADULT` | Excluded (`issp` adults in household): omitted per user review. |
| `H_SETTLEMENT` | Excluded (`wvs7` settlement type): non-US administrative wording duplicating `town_size` and `urban_rural`; omitted per user review. |


In [7]:
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import pandas as pd

CODE_DIR = Path("../..").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona_loaders
importlib.reload(persona_loaders)

def show_value_counts(df, col):
    present_ds = [
        ds for ds in dataset_order
        if df.loc[df["dataset"] == ds, col].notna().any()
    ]
    sub = df.loc[df["dataset"].isin(present_ds), ["dataset", col]].copy()
    sub[col] = sub[col].where(sub[col].notna(), "<NA>").astype(str)

    table = pd.crosstab(sub[col], sub["dataset"], margins=True)
    table = table[present_ds + ["All"]]

    has_na = "<NA>" in table.index
    drop_idx = ["All", "<NA>"] if has_na else ["All"]
    body = table.drop(index=drop_idx).sort_values("All", ascending=False)
    na_row = table.loc[["<NA>"]] if has_na else pd.DataFrame(0, index=["<NA>"], columns=table.columns)
    return pd.concat([body, na_row, table.loc[["All"]]])


all_personas = persona_loaders.load_raw_all_personas()
dataset_order = list(persona_loaders.RAW_LOADERS.keys())
persona_cols = [c for c in all_personas.columns if c not in {"dataset", "CASEID"}]
ds_counts = {c: int(all_personas.loc[all_personas[c].notna(), "dataset"].nunique()) for c in persona_cols}
shared_cols = [c for c, n in ds_counts.items() if n >= 2]
unique_cols = [c for c, n in ds_counts.items() if n == 1]

print(f"Stacked all_personas: {len(all_personas):,} rows x {all_personas.shape[1]} cols")
print(f"Shared ({len(shared_cols)}): {shared_cols}")
print(f"Unique ({len(unique_cols)}): {unique_cols}")
all_personas["dataset"].value_counts().reindex(dataset_order)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Stacked all_personas: 49,468 rows x 90 cols
Shared (29): ['country', 'gender', 'age', 'region', 'family_income', 'religion', 'religious_attendance', 'political_party', 'political_ideology', 'marital_status', 'employment_status', 'household_size', 'urban_rural', 'voted_last_election', 'union_member', 'employer_sector', 'birth_year', 'spouse_employment', 'education', 'interview_language', 'spouse_education', 'race', 'citizen', 'state', 'income', 'hispanic', 'children_in_household', 'nutrition_label_use', 'residence_type']
Unique (59): ['EDUCYRS', 'US_DEGR', 'personal_income', 'WORK', 'WRKHRS', 'EMPREL', 'WRKSUP', 'NSUP', 'TYPORG1', 'occupation', 'PARTLIV', 'SPWORK', 'SPWRKHRS', 'SPEMPREL', 'SPWRKSUP', 'SPISCO08', 'HHCHILDR', 'HHTODD', 'TOPBOT', 'ancestry', 'HHADULT', 'town_size', 'H_SETTLEMENT', 'X003R', 'Q263', 'Q264', 'Q265', 'Q266', 'Q267', 'Q268', 'Q271', 'Q272', 'children_ever_had', 'Q277', 'Q278

dataset
issp               14365
twin2k              2058
wvs7                2596
subpop                22
covid               1971
grummon2023         7945
grummon2025         5636
berke2023           3715
kriner2026          1000
wu2025              1862
kantorowicz2026     2242
magistro2025        6056
Name: count, dtype: int64

## 1. Value counts — shared persona columns

In [3]:
show_value_counts(all_personas, "country")

dataset,issp,twin2k,wvs7,subpop,covid,grummon2023,grummon2025,berke2023,kriner2026,wu2025,kantorowicz2026,magistro2025,All
United States,0,2058,2596,22,1971,7945,5636,3715,1000,1862,2242,0,29047
US,14365,0,0,0,0,0,0,0,0,0,0,3025,17390
Canada,0,0,0,0,0,0,0,0,0,0,0,3031,3031
<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0
All,14365,2058,2596,22,1971,7945,5636,3715,1000,1862,2242,6056,49468


In [4]:
show_value_counts(all_personas, "gender")

dataset,issp,twin2k,wvs7,subpop,covid,grummon2023,grummon2025,berke2023,kriner2026,wu2025,kantorowicz2026,magistro2025,All
gender,,,,,,,,,,,,,
Woman,0,0,0,0,0,4078,3400,0,0,0,0,3277,10755
Female,2197,1044,1206,1,999,0,0,1812,512,973,1127,0,9871
Male,1843,1014,1390,1,949,0,0,1823,483,877,1084,0,9464
Man,0,0,0,0,0,3779,2111,0,0,0,0,2699,8589
2. Female,5709,0,0,0,0,0,0,0,0,0,0,0,5709
1. Male,4586,0,0,0,0,0,0,0,0,0,0,0,4586
Non-binary,0,0,0,0,0,69,112,0,0,0,0,0,181
Other,0,0,0,0,0,0,0,68,0,0,0,79,147
-9. No answer,30,0,0,0,0,0,0,0,0,0,0,0,30


In [5]:
# show_value_counts(all_personas, "age")

In [6]:
show_value_counts(all_personas, "region")

dataset,issp,twin2k,subpop,kriner2026,magistro2025,All
region,,,,,,
5. South Atlantic,2128,0,0,0,0,2128
3. East North Central,1730,0,0,0,0,1730
South,0,0,1,376,1212,1589
9. Pacific,1446,0,0,0,0,1446
2. Middle Atlantic,1144,0,0,0,0,1144
7. West South Central,1108,0,0,0,0,1108
West,0,0,0,235,689,924
"South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)",0,834,0,0,0,834
8. Mountain,817,0,0,0,0,817


In [7]:
# show_value_counts(all_personas, "state")

In [8]:
show_value_counts(all_personas, "education")

dataset,issp,twin2k,wvs7,subpop,covid,grummon2023,grummon2025,kriner2026,wu2025,kantorowicz2026,magistro2025,All
education,,,,,,,,,,,,
Bachelor's degree,0,0,0,0,0,1935,2215,0,0,0,0,4150
Some college or technical school,0,0,0,0,0,1814,1195,0,0,0,0,3009
High school graduate (or GED),0,0,0,0,0,1872,624,0,0,0,0,2496
3. Upper secondary,2391,0,0,0,0,0,0,0,0,0,0,2391
HS or less,0,0,0,0,0,0,0,0,0,0,2374,2374
...,...,...,...,...,...,...,...,...,...,...,...,...
-9. No answer,7,0,0,0,0,0,0,0,0,0,0,7
No answer,2,0,0,0,0,0,0,0,0,0,0,2
9. No answer,1,0,0,0,0,0,0,0,0,0,0,1


In [9]:
# show_value_counts(all_personas, "family_income")

In [10]:
show_value_counts(all_personas, "income")

dataset,subpop,covid,kriner2026,kantorowicz2026,magistro2025,All
income,,,,,,
Middle,0,0,0,0,2996,2996
Higher income,0,0,0,1324,0,1324
High,0,0,0,0,1173,1173
Low,0,0,0,0,1102,1102
Lower income,0,0,0,789,0,789
"$100,000 or more",1,459,202,0,0,662
"$20,000 - $39,999",0,376,161,0,0,537
"$40,000 - $59,999",0,347,169,0,0,516
"Below $20,000",0,328,176,0,0,504


In [11]:
# show_value_counts(all_personas, "race")


In [12]:
show_value_counts(all_personas, "hispanic")

dataset,grummon2023,grummon2025,wu2025,All
hispanic,,,,
No,6647,5049,0,11696
Yes,1297,580,0,1877
None of these,0,0,1629,1629
"Spanish, Hispanic, or Latino",0,0,199,199
<NA>,1,7,34,42
All,7945,5636,1862,15443


In [13]:
# show_value_counts(all_personas, "religion")

In [14]:
# show_value_counts(all_personas, "religious_attendance")

In [15]:
# show_value_counts(all_personas, "political_party")

In [16]:
# show_value_counts(all_personas, "political_ideology")

In [17]:
# show_value_counts(all_personas, "marital_status")

In [18]:
# show_value_counts(all_personas, "employment_status")

In [19]:
# show_value_counts(all_personas, "household_size")

In [20]:
# show_value_counts(all_personas, "children_in_household")

In [21]:
# show_value_counts(all_personas, "children_ever_had")

In [22]:
# show_value_counts(all_personas, "urban_rural")

In [23]:
# show_value_counts(all_personas, "voted_last_election")

In [24]:
# show_value_counts(all_personas, "union_member")

In [25]:
# show_value_counts(all_personas, "residence_type")

In [26]:
# show_value_counts(all_personas, "citizen")

In [27]:
# show_value_counts(all_personas, "interview_language")

In [28]:
# show_value_counts(all_personas, "employer_sector")

In [29]:
# show_value_counts(all_personas, "birth_year") # exclude

In [30]:
# show_value_counts(all_personas, "spouse_education")

In [31]:
# show_value_counts(all_personas, "spouse_employment")

In [32]:
# show_value_counts(all_personas, "nutrition_label_use")

## 2. Value counts — single-dataset persona columns

### 2.1 ISSP (20)

In [33]:
# show_value_counts(all_personas, "EDUCYRS")  # Years of full-time schooling
# exclude

In [34]:
# show_value_counts(all_personas, "ancestry")  # Merged US_ETHN1 + US_ETHN2 ancestry

In [35]:
# show_value_counts(all_personas, "personal_income")  # Personal income (US_RINC)

In [36]:
# show_value_counts(all_personas, "TOPBOT")  # Top-bottom social standing (1-10)

In [37]:
# show_value_counts(all_personas, "WORK")  # Currently, formerly, or never in paid work
# exclude

In [38]:
# show_value_counts(all_personas, "WRKHRS").head(20)  # Hours worked weekly
# exclude

In [39]:
# show_value_counts(all_personas, "EMPREL")  # Employment relationship
# Exclude


In [40]:
# show_value_counts(all_personas, "WRKSUP")  # Supervises other employees
# Exclude

In [41]:
# show_value_counts(all_personas, "NSUP")  # Number of employees supervised
# Exclude


In [42]:
# show_value_counts(all_personas, "TYPORG1")  # For-profit vs non-profit organization
# Exclude


In [43]:
# show_value_counts(all_personas, "occupation")  # Occupation (ISCO-08)

In [44]:
# show_value_counts(all_personas, "PARTLIV")  # Living in steady partnership
# Exclude


In [45]:
# show_value_counts(all_personas, "SPWORK")  # Spouse/partner: paid work history
# Exclude


In [46]:
# show_value_counts(all_personas, "SPWRKHRS")  # Spouse/partner: hours worked weekly
# Exclude


In [47]:
# show_value_counts(all_personas, "SPEMPREL")  # Spouse/partner: employment relationship
# Exclude


In [48]:
# show_value_counts(all_personas, "SPWRKSUP")  # Spouse/partner: supervises other employees
# Exclude


In [49]:
# show_value_counts(all_personas, "SPISCO08")  # Spouse/partner: occupation (ISCO-08)
# Exclude


In [50]:
# show_value_counts(all_personas, "HHCHILDR")  # School-age children in household
# Exclude


In [51]:
# show_value_counts(all_personas, "HHTODD")  # Toddlers below school entry age in household
# Exclude


In [52]:
# show_value_counts(all_personas, "HHADULT")  # Adults in household
# Exclude


### 2.2 WVS7 (21)

In [3]:
# show_value_counts(all_personas, "town_size")  # Town population size (G_TOWNSIZE)

dataset,wvs7,All
G_TOWNSIZE,,
"20,000-50,000",635,635
"100,000-500,000",569,569
"500,000 and more",385,385
"50,000-100,000",239,239
<NA>,768,768
All,2596,2596


In [ ]:
# show_value_counts(all_personas, "H_SETTLEMENT")  # Settlement type
# exclude

dataset,wvs7,All
"Another city, town (not a regional or district center)",1511,1511
District center,954,954
Village,131,131
<NA>,0,0
All,2596,2596


In [55]:
show_value_counts(all_personas, "Q263")  # Respondent immigrant status

dataset,wvs7,All
Q263,,
I am born in this country,2268,2268
I am an immigrant to this country (born outside this country),268,268
<NA>,60,60
All,2596,2596


In [8]:
# show_value_counts(all_personas, "Q264")  # Mother immigrant status
# exclude


In [9]:
# show_value_counts(all_personas, "Q265")  # Father immigrant status
# exclude


In [10]:
# show_value_counts(all_personas, "Q266")  # Country of birth: respondent
# exclude


In [11]:
# show_value_counts(all_personas, "Q267")  # Country of birth: mother
# exclude


In [12]:
# show_value_counts(all_personas, "Q268")  # Country of birth: father
# exclude


In [ ]:
show_value_counts(all_personas, "Q271")  # Lives with parents


dataset,wvs7,All
Q271,,
No,2317,2317
"Yes, own parent(s)",233,233
"Yes, parent(s) in law",24,24
"Yes, both own parent(s) and parent(s) in law",4,4
<NA>,18,18
All,2596,2596


In [13]:
show_value_counts(all_personas, "Q272")  # Language spoken at home

dataset,wvs7,All
Q272,,
English,2413,2413
Spanish; Castilian,116,116
Other,32,32
Standard Chinese; Mandarin; Putonghua; Guoyu,21,21
Vietnamese,3,3
Japanese,1,1
<NA>,10,10
All,2596,2596


In [14]:
# show_value_counts(all_personas, "Q277")  # Mother highest education
# exclude


In [15]:
# show_value_counts(all_personas, "Q278")  # Father highest education
# exclude


In [17]:
show_value_counts(all_personas, "Q281")  # Respondent occupational group


dataset,wvs7,All
Q281,,
"Professional and technical (for example: doctor, teacher, engineer, artist, accountant, nurse)",580,580
"Service (for example: restaurant owner, police officer, waitress, barber, caretaker)",187,187
"Clerical (for example: secretary, clerk, office manager, civil servant, bookkeeper",168,168
"Sales (for example: sales manager, shop owner, shop assistant, insurance agent, buyer)",154,154
"Skilled worker (for example: foreman, motor mechanic, printer, seamstress, tool and die maker, electrician)",145,145
"Higher administrative (for example: banker, executive in big business, high government official, union official)",102,102
"Unskilled worker (for example: labourer, porter, unskilled factory worker, cleaner)",80,80
"Semi-skilled worker (for example: bricklayer, bus driver, cannery worker, carpenter, sheet metal worker, baker)",68,68
"JP,KG,TJ: Other",53,53


In [19]:
# show_value_counts(all_personas, "Q282")  # Spouse occupational group
# exclude


In [20]:
# show_value_counts(all_personas, "Q283")  # Father occupational group (at age 14)
# exclude


In [21]:
show_value_counts(all_personas, "Q285")  # Chief wage earner in household

dataset,wvs7,All
Q285,,
Yes,1487,1487
No,1083,1083
<NA>,26,26
All,2596,2596


In [ ]:
show_value_counts(all_personas, "Q286")  # Family savings during past year


dataset,wvs7,All
Q286,,
Save money,1081,1081
Just get by,973,973
Spent some savings and borrowed money,314,314
Spent savings and borrowed money,193,193
<NA>,35,35
All,2596,2596


In [70]:
show_value_counts(all_personas, "Q287")  # Subjective social class

dataset,wvs7,All
Q287,,
Lower middle class,967,967
Upper middle class,762,762
Working class,615,615
Lower class,183,183
Upper class,34,34
<NA>,35,35
All,2596,2596


In [71]:
show_value_counts(all_personas, "Q288")  # Scale of household incomes (1-10)

dataset,wvs7,All
Q288,,
Fifth step,599,599
Sixth step,468,468
Fourth step,373,373
Seventh step,370,370
Third step,268,268
Eight step,151,151
second step,128,128
Lower step,123,123
Nineth step,36,36


In [22]:
# show_value_counts(all_personas, "Q289CS9")  # Detailed religious denomination
# exclude


In [73]:
show_value_counts(all_personas, "voting_frequency")  # National election voting frequency (Q222)

dataset,wvs7,All
voting_frequency,,
Always,1582,1582
Usually,546,546
Never,356,356
Not allowed to vote,78,78
<NA>,34,34
All,2596,2596


### 2.3 Grummon2023 (5)

In [74]:
show_value_counts(all_personas, "DIETQUAL")  # Self-rated diet quality

dataset,grummon2023,All
Good,3216,3216
Fair,2356,2356
Very good,1463,1463
Excellent,465,465
Poor,445,445
<NA>,0,0
All,7945,7945


In [75]:
show_value_counts(all_personas, "MEDHELP")  # Help reading medical materials (health literacy)

dataset,grummon2023,All
Never,4201,4201
Rarely,1908,1908
Sometimes,1224,1224
Most of the time,382,382
Always,230,230
<NA>,0,0
All,7945,7945


In [76]:
show_value_counts(all_personas, "VENDING_BEV")  # Vending machine beverage frequency

dataset,grummon2023,All
Never or less than 1 time per week,5548,5548
1 to 2 times per week,1613,1613
3 to 4 times per week,592,592
5 or more times per week,192,192
<NA>,0,0
All,7945,7945


In [77]:
show_value_counts(all_personas, "VENDING_SNACK")  # Vending machine snack frequency

dataset,grummon2023,All
VENDING_SNACK,,
Never or less than 1 time per week,5656,5656
1 to 2 times per week,1470,1470
3 to 4 times per week,615,615
5 or more times per week,181,181
<NA>,23,23
All,7945,7945


In [78]:
show_value_counts(all_personas, "BMI")  # Body mass index category

dataset,grummon2023,All
BMI,,
Healthy weight (18.5 to <25.0),2720,2720
Obese (>=30.0),2514,2514
Overweight (25.0 to <30.0),2385,2385
Underweight (<18.5),312,312
<NA>,14,14
All,7945,7945


### 2.4 Grummon2025 (4)

In [79]:
show_value_counts(all_personas, "SNAP")  # Household receives SNAP

dataset,grummon2025,All
SNAP,,
No,4773,4773
Yes,817,817
Don't know,29,29
<NA>,17,17
All,5636,5636


In [80]:
show_value_counts(all_personas, "WIC")  # Household receives WIC

dataset,grummon2025,All
WIC,,
No,5434,5434
Yes,138,138
Don't know,47,47
<NA>,17,17
All,5636,5636


In [81]:
show_value_counts(all_personas, "ONLINEAMT")  # Share of groceries bought online

dataset,grummon2025,All
ONLINEAMT,,
Less than 25%,3136,3136
25-49%,991,991
75% or more,808,808
50-74%,685,685
<NA>,16,16
All,5636,5636


In [82]:
show_value_counts(all_personas, "HEALTHLIT")  # Health literacy score (0-4)

dataset,grummon2025,All
4 out of 4,3257,3257
3 out of 4,1361,1361
2 out of 4,544,544
1 out of 4,327,327
0 out of 4,147,147
<NA>,0,0
All,5636,5636


### 2.5 Wu2025 (6)

In [83]:
show_value_counts(all_personas, "POL_LABEL")  # Categorical political label (ideology_original)

dataset,wu2025,All
POL_LABEL,,
Conservative,672,672
Liberal,536,536
Other,299,299
Centrist,125,125
Libertarian,122,122
Socialist,76,76
Communist,26,26
<NA>,6,6
All,1862,1862


In [84]:
show_value_counts(all_personas, "OWNHOME")  # Owns residence

dataset,wu2025,All
OWNHOME,,
Yes,1193,1193
No,667,667
<NA>,2,2
All,1862,1862


In [85]:
show_value_counts(all_personas, "RENTHOME")  # Rents residence

dataset,wu2025,All
RENTHOME,,
No,1116,1116
Yes,745,745
<NA>,1,1
All,1862,1862


In [86]:
show_value_counts(all_personas, "ORIENTATION")  # Sexual orientation

dataset,wu2025,All
ORIENTATION,,
Heterosexual (straight),1643,1643
Bisexual,110,110
Homosexual (gay),59,59
Prefer not to say,24,24
Other,23,23
<NA>,3,3
All,1862,1862


In [87]:
show_value_counts(all_personas, "MILITARY")  # Military/veteran status

dataset,wu2025,All
MILITARY,,
No,1586,1586
Yes,261,261
<NA>,15,15
All,1862,1862


In [88]:
show_value_counts(all_personas, "ELECHEAT")  # Uses electricity to heat home

dataset,wu2025,All
ELECHEAT,,
Yes,1140,1140
No,668,668
<NA>,54,54
All,1862,1862
